# In-Situ Temperature-Resolved XRD Analysis Pipeline

An automated workflow for tracking structural phase transitions, lattice thermal expansion, and peak deconvolution in MXene and MAX-phase materials (e.g., $\text{Ti}_2\text{AlC}_{0.5}\text{N}_{0.5} \rightarrow \text{TiO}_2$ Rutile).

### Workflow Overview:
1. **Setup & Config**: Dynamic path resolution and experimental parameters.
2. **Crystallographic References**: ICDD/PDF reference lines with thermal lattice shift.
3. **Data Loading & Peak Tracking**: Scan  files, parse temperatures, and locate peak positions.
4. **Visualizations**: 1D Diffractogram Stack & 2D Intensity Heatmap.
5. **Animation & Presentation**: Dynamic peak labeling, animated GIFs, and PowerPoint export.


## 1. Imports and Environment Setup

In [ ]:
from __future__ import print_function, division

import os
import re
import glob
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors, gridspec
import matplotlib.patheffects as pe
from matplotlib.ticker import MultipleLocator

%matplotlib inline

try:
    from IPython.display import Image as IPyImage, display
except ImportError:
    IPyImage = display = None

# Ensure local project root is on sys.path to load config.py
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import config

print("Environment initialized successfully.")


## 2. Experimental Parameters & Paths
All parameters are loaded from `config.py`. You can adjust them directly in  or override them below.


In [ ]:
INPUT_DIR = str(config.DATA_DIR)
OUTPUT_FILE = str(config.OUTPUT_DIR / "analysis")
FILE_PATTERN = config.FILE_PATTERN
COMMENT_CHAR = config.COMMENT_CHAR
TEMP_PATTERN = config.TEMP_PATTERN
TEMP_UNIT = config.TEMP_UNIT

# Angle windows (2theta in degrees)
TWO_THETA_CROP_MIN = config.TWO_THETA_CROP_MIN
TWO_THETA_MIN = config.TWO_THETA_MIN
TWO_THETA_MAX = config.TWO_THETA_MAX

# Visualization & Animation dials
PLOT_2THETA_MIN = config.PLOT_2THETA_MIN
PLOT_2THETA_MAX = config.PLOT_2THETA_MAX
WATERFALL_OFFSET = config.WATERFALL_OFFSET
SHOW_PEAK_TRACK = config.SHOW_PEAK_TRACK
CMAP_NAME = config.CMAP_NAME

MAKE_GIF = config.MAKE_GIF
SPLIT_TEMP = config.SPLIT_TEMP
GIF_CUMULATIVE = config.GIF_CUMULATIVE
GIF_STEP = config.GIF_STEP
GIF_FPS = config.GIF_FPS
GIF_DPI = config.GIF_DPI
GIF_HOLD_LAST_S = config.GIF_HOLD_LAST_S
GIF_SHARED_COLORS = config.GIF_SHARED_COLORS
GIF_LOOP = config.GIF_LOOP
MAKE_MP4 = config.MAKE_MP4
MAKE_PPTX = config.MAKE_PPTX

print(f"Data Directory:      {INPUT_DIR}")
print(f"Output Prefix:       {OUTPUT_FILE}")
print(f"Peak Tracking Range: [{TWO_THETA_MIN}, {TWO_THETA_MAX}] deg")


## 3. Data Ingestion & Export Helper Functions

In [ ]:
def extract_temp(filename):
    """Extract numerical temperature from filename using regex."""
    match = TEMP_PATTERN.search(filename)
    if match is None:
        print("[{0}] warning: could not extract temperature from filename".format(filename))
        return np.nan
    return float(match.group(1))

def read_data(path):
    """Read 2-column .xy diffraction data."""
    try:
        data = np.loadtxt(path, comments=COMMENT_CHAR, ndmin=2)
    except ValueError:
        data = np.loadtxt(path, comments=COMMENT_CHAR, delimiter=",", ndmin=2)
    if data.shape[1] < 2:
        raise ValueError("File must contain at least 2 columns (2theta, intensity)")
    return data

def peak_in_range(two_theta, intensity, lo, hi, name=""):
    """Find peak position in angle window."""
    mask = (two_theta >= lo) & (two_theta <= hi)
    if not mask.any():
        print("[{0}] warning: no points found in range [{1}, {2}] deg".format(name, lo, hi))
        return np.nan
    return two_theta[mask][np.argmax(intensity[mask])]

def write_gif(path, arrays, fps):
    """Encode frames to animated GIF via Pillow or imageio."""
    try:
        import PIL
        from PIL import Image
        images = [Image.fromarray(a) for a in arrays]
        images[0].save(path, save_all=True, append_images=images[1:],
                       duration=int(1000.0 / fps),
                       loop=0 if GIF_LOOP else 1)
        return "PIL"
    except ImportError:
        pass
    try:
        import imageio
        imageio.mimsave(path, arrays, fps=fps,
                        loop=0 if GIF_LOOP else 1)
        return "imageio"
    except ImportError:
        raise ImportError("Serve Pillow o imageio: pip install Pillow imageio")

def write_mp4(path, arrays, fps):
    """Encode frames to MP4 video."""
    import imageio
    writer = imageio.get_writer(path, fps=fps, codec="libx264", quality=8)
    for frame in arrays:
        writer.append_data(frame)
    writer.close()

def build_pptx(path, slides):
    """Build PowerPoint presentation deck from images."""
    import pptx
    from pptx.util import Inches, Pt
    prs = pptx.Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    blank_layout = prs.slide_layouts[6]
    for title, img_path in slides:
        slide = prs.slides.add_slide(blank_layout)
        tb = slide.shapes.add_textbox(Inches(0.8), Inches(0.4), Inches(11.7), Inches(0.8))
        p = tb.text_frame.paragraphs[0]
        p.text = title
        p.font.size = Pt(24)
        p.font.bold = True
        slide.shapes.add_picture(img_path, Inches(0.8), Inches(1.3), height=Inches(5.8))
    prs.save(path)


## 4. Crystallographic Reference Calculations
Calculates thermal shifts for reference Bragg reflections based on lattice expansion:
24674\theta(T) = \arcsin \left( \frac{\sin\theta_0}{1 + \alpha(T - T_0)} \right)24674


In [ ]:
SHOW_REFS = config.SHOW_REFS
LAMBDA_CU_KA1 = config.LAMBDA_CU_KA1
LABEL_MIN_I = config.LABEL_MIN_I
MIN_LABEL_SEP = config.MIN_LABEL_SEP
REF_T_C = config.REF_T_C
PHASES = config.PHASES

def shifted_two_theta(tt0, temp, alpha):
    """Calculates thermal lattice shift for Bragg peak."""
    if not np.isfinite(temp) or alpha == 0:
        return tt0
    rad0 = np.radians(tt0 / 2.0)
    sin_new = np.sin(rad0) / (1.0 + alpha * (temp - REF_T_C))
    sin_new = np.clip(sin_new, -1.0, 1.0)
    return np.degrees(2.0 * np.arcsin(sin_new))

def hkl_label(h, k, l):
    parts = []
    for v in (h, k, l):
        parts.append(r"ar{{{0}}}".format(abs(v)) if v < 0 else str(v))
    return r"$".format("".join(parts))

def draw_refs(ax, lo, hi, temp):
    """Draws reference ICDD bars and hkl annotations on axis."""
    ax.set_xlim(lo, hi)
    ax.set_ylim(-0.5, len(PHASES) * 1.0)
    ax.set_yticks([])
    ax.set_xlabel("2theta (deg)")
    placed = []
    for idx_p, ph in enumerate(PHASES):
        base = idx_p * 1.0
        color = ph.get("color", "k")
        alpha = ph.get("alpha", 0.0)
        pks = [(tt, inten, h, k, l) for (tt, inten, h, k, l) in ph["peaks"]
               if lo <= shifted_two_theta(tt, temp, alpha) <= hi]
        if not pks:
            continue
        max_i = max(p[1] for p in pks) or 1.0
        for tt0, raw_i, h, k, l in pks:
            x = shifted_two_theta(tt0, temp, alpha)
            h_rel = (raw_i / float(max_i)) * 0.75
            ax.vlines(x, base, base + h_rel, color=color, linewidth=1.2)
            if raw_i >= LABEL_MIN_I and not any(abs(x - p) < MIN_LABEL_SEP * (hi - lo) for p in placed):
                lbl = hkl_label(h, k, l)
                ax.text(x, base + h_rel + 0.03, lbl, ha="center", va="bottom",
                        fontsize=6, color=color, rotation=90)
                placed.append(x)
        ax.text(hi, base + 0.98, ph["name"], ha="right", va="top", fontsize=7,
                color=ph["color"], fontweight="bold")

print(f"Loaded {len(PHASES)} reference crystallographic phases.")


## 5. Ingestion of `.xy` Scans & Peak Tracking

In [ ]:
files = sorted(glob.glob(os.path.join(INPUT_DIR, FILE_PATTERN)))
if not files:
    print("No files matching {0} found in {1}".format(FILE_PATTERN, INPUT_DIR))
    print("Tip: Run 'python scripts/generate_synthetic_xrd.py' to generate demo data.")

results = []
patterns = []
peak_by_name = {}

print(f"Scanning {len(files)} files...")
for f in files:
    name = os.path.basename(f)
    temp = extract_temp(name)
    try:
        data = read_data(f)
    except Exception as e:
        print("[{0}] error: {1}".format(name, e))
        continue
    if TWO_THETA_CROP_MIN is not None:
        data = data[data[:, 0] >= TWO_THETA_CROP_MIN]
        if data.shape[0] == 0:
            print("[{0}] warning: no points left above {1} deg".format(name, TWO_THETA_CROP_MIN))
            results.append((temp, np.nan))
            continue
    tt, inten = data[:, 0], data[:, 1]
    peak = peak_in_range(tt, inten, TWO_THETA_MIN, TWO_THETA_MAX, name)
    results.append((temp, peak))
    patterns.append((temp, tt, inten, name))
    peak_by_name[name] = peak
    print("  [{0}] T={1:4.0f} °C -> Peak at 2theta={2:.4f}°".format(name, temp, peak))


## 6. Export Summary Table

In [ ]:
if results:
    out_dir = os.path.dirname(OUTPUT_FILE)
    if out_dir and not os.path.isdir(out_dir):
        os.makedirs(out_dir)

    table = np.array(results, dtype=float)
    table = table[np.argsort(table[:, 0], kind="mergesort")]

    header = (
        f"# Peak position (max intensity) in range [{TWO_THETA_MIN}, {TWO_THETA_MAX}] deg vs temperature\n"
        "# columns: temperature  peak_position_2theta"
    )
    table_path = OUTPUT_FILE + "_peak_positions.txt"
    np.savetxt(table_path, table, fmt="%8.2f  %10.4f", header=header)
    print(f"Saved summary table: {table_path}")
    
    # Preview top 5 rows
    print("\n--- Preview of Results Table ---")
    print("Temperature (°C) | Peak Position (2θ deg)")
    for t_val, p_val in table[:5]:
        print(f"{t_val:15.1f} | {p_val:10.4f}")


## 7. Visualization 1: 1D Diffractograms Stack (Overlay)

In [ ]:
plot_pats = sorted([p for p in patterns if np.isfinite(p[0])], key=lambda p: p[0])

if not plot_pats:
    print("No valid diffractograms to plot.")
else:
    temps = np.array([p[0] for p in plot_pats])
    lo = PLOT_2THETA_MIN if PLOT_2THETA_MIN is not None else min(p[1].min() for p in plot_pats)
    hi = PLOT_2THETA_MAX if PLOT_2THETA_MAX is not None else max(p[1].max() for p in plot_pats)

    try:
        cmap = plt.get_cmap(CMAP_NAME)
    except ValueError:
        cmap = plt.get_cmap("jet")
    norm = colors.Normalize(vmin=temps.min(), vmax=temps.max())
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    max_int = 0.0
    for temp, tt, inten, name in plot_pats:
        m = (tt >= lo) & (tt <= hi)
        if m.any():
            max_int = max(max_int, inten[m].max())
    step = WATERFALL_OFFSET * max_int

    fig1, ax1 = plt.subplots(figsize=(10, 6))
    for i, (temp, tt, inten, name) in enumerate(plot_pats):
        ax1.plot(tt, inten + i * step, color=cmap(norm(temp)), lw=1.2, alpha=0.85)

    ax1.set_xlim(lo, hi)
    ax1.set_xlabel("2theta (deg)")
    ax1.set_ylabel("Intensity (a.u.)")
    ax1.set_title("Diffractograms Stack ({0} scans)".format(len(plot_pats)))
    fig1.colorbar(sm, ax=ax1, label="Temperature ({0})".format(TEMP_UNIT))
    fig1.tight_layout()
    overlay_path = OUTPUT_FILE + "_overlay.png"
    fig1.savefig(overlay_path, dpi=200)
    static_slides = [("Diffractograms", overlay_path)]
    print("Saved plot: {0}".format(overlay_path))
    plt.show()


## 8. Visualization 2: 2D Intensity Heatmap & Peak Tracking

In [ ]:
if len(plot_pats) >= 2:
    grid = np.linspace(lo, hi, 1000)
    rows = []
    for temp, tt, inten, name in plot_pats:
        order = np.argsort(tt)
        rows.append(np.interp(grid, tt[order], inten[order],
                              left=np.nan, right=np.nan))
    Z = np.ma.masked_invalid(np.vstack(rows))

    fig2, ax2 = plt.subplots(figsize=(10, 6))
    mesh = ax2.pcolormesh(grid, temps, Z, cmap=cmap, shading="gouraud",
                          vmin=0, vmax=np.percentile(Z.compressed(), 99.5))
    fig2.colorbar(mesh, ax=ax2, label="Intensity (a.u.)")

    if SHOW_PEAK_TRACK:
        t = table[np.isfinite(table[:, 0]) & np.isfinite(table[:, 1])]
        ax2.plot(t[:, 1], t[:, 0], color="red", lw=2, linestyle="--", label="Peak track")
        ax2.legend(loc="upper right")

    ax2.set_xlim(lo, hi)
    ax2.set_xlabel("2theta (deg)")
    ax2.set_ylabel("Temperature ({0})".format(TEMP_UNIT))
    ax2.set_title("Diffractogram 2D Map (Heatmap)")
    fig2.tight_layout()
    heatmap_path = OUTPUT_FILE + "_heatmap.png"
    fig2.savefig(heatmap_path, dpi=200)
    static_slides.append(("Diffractogram map", heatmap_path))
    print("Saved plot: {0}".format(heatmap_path))
    plt.show()


## 9. Dynamic Peak Labeling & Noise Floor Check

In [ ]:
SHOW_PEAK_LABELS = config.SHOW_PEAK_LABELS
LABEL_FONTSIZE = config.LABEL_FONTSIZE
LABELS_ON_ALL_SHOWN = config.LABELS_ON_ALL_SHOWN
MXENE_LABELS = config.MXENE_LABELS
MXENE_LABEL_COLOR = config.MXENE_LABEL_COLOR
MXENE_PRESENCE_SIGMA = config.MXENE_PRESENCE_SIGMA
RUTILE_LABELS = config.RUTILE_LABELS
RUTILE_LABEL_COLOR = config.RUTILE_LABEL_COLOR
RUTILE_PRESENCE_SIGMA = config.RUTILE_PRESENCE_SIGMA

def window_peak(tt, inten, w_lo, w_hi, sigma=0.0):
    info = {"reason": "", "x": np.nan, "height": 0.0, "required": 0.0}
    idx = np.where((tt >= w_lo) & (tt <= w_hi))[0]
    if len(idx) < 3:
        info["reason"] = "nessun dato nella finestra"
        return None, info
    i_max = idx[np.argmax(inten[idx])]
    x_m, y_m = tt[i_max], inten[i_max]
    info["x"] = x_m
    if sigma > 0:
        width = w_hi - w_lo
        left = np.where((tt >= w_lo - width) & (tt < w_lo))[0]
        right = np.where((tt > w_hi) & (tt <= w_hi + width))[0]
        wide = np.where((tt >= w_lo - width) & (tt <= w_hi + width))[0]
        if i_max == idx[0] or i_max == idx[-1]:
            info["reason"] = "il massimo e' sul bordo della finestra (non c'e' un picco)"
            return None, info
        if len(left) >= 3 and len(right) >= 3:
            xl, yl = np.median(tt[left]), np.median(inten[left])
            xr, yr = np.median(tt[right]), np.median(inten[right])
            base = yl + (yr - yl) * (x_m - xl) / (xr - xl)
        else:
            base = np.median(inten[wide])
        d = np.diff(inten[wide])
        noise = 1.4826 * np.median(np.abs(d - np.median(d))) / np.sqrt(2.0)
        info["height"] = y_m - base
        info["required"] = sigma * noise
        if info["height"] <= 0 or info["height"] < info["required"]:
            info["reason"] = "troppo basso rispetto al rumore"
            return None, info
    return (x_m, y_m), info

def build_label_specs(with_rutile):
    specs = []
    for tag, w_lo, w_hi in MXENE_LABELS:
        specs.append({"key": "mx_" + tag, "text": "MXene (" + tag + ")",
                      "w_lo": w_lo, "w_hi": w_hi,
                      "color": MXENE_LABEL_COLOR,
                      "sigma": MXENE_PRESENCE_SIGMA})
    if with_rutile:
        for tag, center, half in RUTILE_LABELS:
            specs.append({"key": "ru_" + tag, "text": "Rutile (" + tag + ")",
                          "w_lo": center - half, "w_hi": center + half,
                          "color": RUTILE_LABEL_COLOR,
                          "sigma": RUTILE_PRESENCE_SIGMA})
    return specs


## 10. In-Situ Animation Generation (GIF / MP4)

In [ ]:
def make_gif(pats, gif_path, cmap, lo, hi, peak_by_name, tag, norm_global, label_specs=()):
    idxs = list(range(0, len(pats), GIF_STEP))
    if not idxs:
        idxs = [0]
    elif idxs[-1] != len(pats) - 1:
        idxs.append(len(pats) - 1)

    t_sub = np.array([pats[i][0] for i in idxs])
    if GIF_SHARED_COLORS:
        norm_g = norm_global
    else:
        norm_g = colors.Normalize(vmin=t_sub.min(), vmax=t_sub.max())

    y_hi = 0.0
    for temp, tt, inten, name in pats:
        m = (tt >= lo) & (tt <= hi)
        if m.any():
            y_hi = max(y_hi, inten[m].max())
    y_hi *= 1.15
    y_lo = 0.0

    frames = []
    if SHOW_REFS:
        fig_g = plt.figure(figsize=(9, 6), dpi=GIF_DPI)
        gs = gridspec.GridSpec(2, 1, height_ratios=[4, 1], hspace=0.08)
        ax_g = fig_g.add_subplot(gs[0])
        ax_r = fig_g.add_subplot(gs[1], sharex=ax_g)
    else:
        fig_g, ax_g = plt.subplots(figsize=(9, 5), dpi=GIF_DPI)
        ax_r = None

    for k, idx_cur in enumerate(idxs):
        temp_k, tt_k, inten_k, name_k = pats[idx_cur]
        ax_g.cla()
        if ax_r is not None:
            ax_r.cla()

        shown = idxs[:k + 1] if GIF_CUMULATIVE else [idx_cur]
        for j, s_idx in enumerate(shown):
            t_s, tt_s, inten_s, name_s = pats[s_idx]
            is_cur = (s_idx == idx_cur)
            col = cmap(norm_g(t_s))
            lw = 1.6 if is_cur else 0.7
            alph = 1.0 if is_cur else 0.35
            ax_g.plot(tt_s, inten_s, color=col, lw=lw, alpha=alph)

        ax_g.set_xlim(lo, hi)
        ax_g.set_ylim(y_lo, y_hi)
        ax_g.set_ylabel("Intensity (a.u.)")
        ax_g.set_title(f"{tag}\nT = {temp_k:.0f} {TEMP_UNIT} ({name_k})", fontsize=10)
        if ax_r is not None:
            ax_g.tick_params(labelbottom=False)
            draw_refs(ax_r, lo, hi, temp_k)
        else:
            ax_g.set_xlabel("2theta (deg)")

        fig_g.canvas.draw()
        rgba = np.asarray(fig_g.canvas.buffer_rgba())
        frames.append(rgba[:, :, :3].copy())
    plt.close(fig_g)

    frames.extend([frames[-1]] * int(round(GIF_HOLD_LAST_S * GIF_FPS)))
    how = write_gif(gif_path, frames, GIF_FPS)
    return how, len(frames), len(idxs), frames

gif_done = []
if MAKE_GIF and plot_pats:
    part1 = [p for p in plot_pats if p[0] <= SPLIT_TEMP]
    part2 = [p for p in plot_pats if p[0] >= SPLIT_TEMP]
    jobs = [
        (part1, OUTPUT_FILE + "_evolution_RT-{0:g}.gif".format(SPLIT_TEMP),
         "Room T to {0:g} {1}".format(SPLIT_TEMP, TEMP_UNIT), False),
        (part2, OUTPUT_FILE + "_evolution_{0:g}-end.gif".format(SPLIT_TEMP),
         "{0:g} {1} to end".format(SPLIT_TEMP, TEMP_UNIT), True),
    ]
    for pats, gif_path, tag, with_rutile in jobs:
        if len(pats) < 2:
            print("GIF non creata ({0}): meno di 2 diffrattogrammi in questo intervallo".format(tag))
            continue
        try:
            how, n_frames, n_pats, frames = make_gif(
                pats, gif_path, cmap, lo, hi, peak_by_name, tag, norm,
                build_label_specs(with_rutile) if SHOW_PEAK_LABELS else [])
            print("Saved: {0} ({1} diffrattogrammi, {2} frames, scritta con {3})".format(
                gif_path, n_pats, n_frames, how))
            gif_done.append((tag, gif_path))
            if IPyImage and display:
                display(IPyImage(filename=gif_path))
        except Exception as e:
            print(f"Could not generate GIF ({tag}): {e}")

## 11. Presentation Slide Export (PowerPoint)

In [ ]:
if MAKE_PPTX:
    pptx_path = OUTPUT_FILE + "_presentation.pptx"
    try:
        build_pptx(pptx_path, static_slides + gif_done)
        print("Saved: {0} ({1} slide)".format(pptx_path, len(static_slides) + len(gif_done)))
    except ImportError:
        print("python-pptx non installato: esegui 'pip install python-pptx' per abilitare")
    except Exception as e:
        print(f"PPTX non creato: {e}")
else:
    print("PowerPoint export is disabled (MAKE_PPTX=False in config.py).")
